In [1]:
# NHL Analytics - API Query Setup
import requests
import pandas as pd
import json
from datetime import datetime
import time

# Base URL for NHL API
BASE_URL = "https://api.nhle.com/stats/rest/en"

def query_nhl_api(endpoint, params=None):
    """
    Query the NHL API with error handling and rate limiting
    
    Args:
        endpoint (str): The API endpoint to query
        params (dict): Query parameters
    
    Returns:
        dict: JSON response from the API
    """
    url = f"{BASE_URL}/{endpoint}"
    
    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        
        # Add small delay to be respectful to the API
        time.sleep(0.1)
        
        return response.json()
    
    except requests.exceptions.RequestException as e:
        print(f"Error querying API: {e}")
        return None

def explore_api_structure():
    """Explore available API endpoints and their structure"""
    print("NHL API Exploration")
    print("=" * 50)
    
    # Test basic connectivity
    config = query_nhl_api("config")
    if config:
        print("✓ API is accessible")
        print(f"Available report types: {list(config.keys())}")
    else:
        print("✗ API is not accessible")
        return
    
    # Explore different data types
    endpoints_to_test = [
        ("franchise", "Franchise information"),
        ("season", "Season information"),
        ("skater/summary", "Skater summary stats"),
        ("team/summary", "Team summary stats"),
        ("goalie/summary", "Goalie summary stats")
    ]
    
    for endpoint, description in endpoints_to_test:
        print(f"\n{description}:")
        data = query_nhl_api(endpoint)
        if data:
            print(f"  ✓ {endpoint} - {len(data.get('data', []))} records")
            if 'data' in data and len(data['data']) > 0:
                print(f"  Sample columns: {list(data['data'][0].keys())[:5]}...")
        else:
            print(f"  ✗ {endpoint} - Failed to retrieve")

# Run the exploration
explore_api_structure()


NHL API Exploration
✓ API is accessible
Available report types: ['playerReportData', 'goalieReportData', 'teamReportData', 'aggregatedColumns', 'individualColumns']

Franchise information:
  ✓ franchise - 40 records
  Sample columns: ['id', 'fullName', 'teamCommonName', 'teamPlaceName']...

Season information:
  ✓ season - 108 records
  Sample columns: ['id', 'allStarGameInUse', 'conferencesInUse', 'divisionsInUse', 'endDate']...

Skater summary stats:
Error querying API: 500 Server Error: Server Error for url: https://api.nhle.com/stats/rest/en/skater/summary
  ✗ skater/summary - Failed to retrieve

Team summary stats:
  ✓ team/summary - 50 records
  Sample columns: ['faceoffWinPct', 'gamesPlayed', 'goalsAgainst', 'goalsAgainstPerGame', 'goalsFor']...

Goalie summary stats:
Error querying API: 500 Server Error: Server Error for url: https://api.nhle.com/stats/rest/en/goalie/summary
  ✗ goalie/summary - Failed to retrieve


In [2]:
# Specific NHL Data Queries

def get_player_stats(season="20232024", report_type="summary", limit=100):
    """
    Get player statistics for a specific season
    
    Args:
        season (str): Season in format YYYY (e.g., "20232024")
        report_type (str): Type of report (summary, bios, etc.)
        limit (int): Number of records to return
    
    Returns:
        pandas.DataFrame: Player statistics
    """
    params = {
        "cayenneExp": f"gameTypeId=2 and seasonId<={season}",
        "limit": limit
    }
    
    data = query_nhl_api(f"skater/{report_type}", params)
    
    if data and 'data' in data:
        df = pd.DataFrame(data['data'])
        print(f"Retrieved {len(df)} player records")
        return df
    else:
        print("Failed to retrieve player data")
        return pd.DataFrame()

def get_team_stats(season="20232024", report_type="summary"):
    """
    Get team statistics for a specific season
    
    Args:
        season (str): Season in format YYYY (e.g., "20232024")
        report_type (str): Type of report (summary, etc.)
    
    Returns:
        pandas.DataFrame: Team statistics
    """
    params = {
        "cayenneExp": f"gameTypeId=2 and seasonId<={season}"
    }
    
    data = query_nhl_api(f"team/{report_type}", params)
    
    if data and 'data' in data:
        df = pd.DataFrame(data['data'])
        print(f"Retrieved {len(df)} team records")
        return df
    else:
        print("Failed to retrieve team data")
        return pd.DataFrame()

def get_goalie_stats(season="20232024", report_type="summary", limit=50):
    """
    Get goalie statistics for a specific season
    
    Args:
        season (str): Season in format YYYY (e.g., "20232024")
        report_type (str): Type of report (summary, etc.)
        limit (int): Number of records to return
    
    Returns:
        pandas.DataFrame: Goalie statistics
    """
    params = {
        "cayenneExp": f"gameTypeId=2 and seasonId<={season}",
        "limit": limit
    }
    
    data = query_nhl_api(f"goalie/{report_type}", params)
    
    if data and 'data' in data:
        df = pd.DataFrame(data['data'])
        print(f"Retrieved {len(df)} goalie records")
        return df
    else:
        print("Failed to retrieve goalie data")
        return pd.DataFrame()

# Example usage - uncomment to run specific queries
print("NHL API Query Functions Ready!")
print("Available functions:")
print("- get_player_stats(season, report_type, limit)")
print("- get_team_stats(season, report_type)")
print("- get_goalie_stats(season, report_type, limit)")
print("\nExample: df = get_player_stats('20232024', 'summary', 50)")


NHL API Query Functions Ready!
Available functions:
- get_player_stats(season, report_type, limit)
- get_team_stats(season, report_type)
- get_goalie_stats(season, report_type, limit)

Example: df = get_player_stats('20232024', 'summary', 50)


In [3]:
# Example Queries and Data Analysis

# Get current season player stats (top 50 players)
print("Fetching current season player statistics...")
player_stats = get_player_stats(season="20232024", report_type="summary", limit=50)

if not player_stats.empty:
    print(f"\nPlayer Stats Shape: {player_stats.shape}")
    print(f"Columns: {list(player_stats.columns)}")
    
    # Show top 10 players by points
    if 'points' in player_stats.columns:
        top_scorers = player_stats.nlargest(10, 'points')[['playerName', 'points', 'goals', 'assists']]
        print("\nTop 10 Scorers:")
        print(top_scorers)
    
    # Show basic statistics
    print(f"\nBasic Stats Summary:")
    numeric_cols = player_stats.select_dtypes(include=['number']).columns
    print(player_stats[numeric_cols].describe())
else:
    print("No player data retrieved")

print("\n" + "="*60)

# Get team stats
print("Fetching current season team statistics...")
team_stats = get_team_stats(season="20232024", report_type="summary")

if not team_stats.empty:
    print(f"\nTeam Stats Shape: {team_stats.shape}")
    print(f"Columns: {list(team_stats.columns)}")
    
    # Show team standings by points
    if 'points' in team_stats.columns:
        standings = team_stats.nlargest(10, 'points')[['teamName', 'points', 'wins', 'losses', 'otLosses']]
        print("\nTop 10 Teams by Points:")
        print(standings)
else:
    print("No team data retrieved")


Fetching current season player statistics...
Retrieved 50 player records

Player Stats Shape: (50, 26)
Columns: ['assists', 'evGoals', 'evPoints', 'faceoffWinPct', 'gameWinningGoals', 'gamesPlayed', 'goals', 'lastName', 'otGoals', 'penaltyMinutes', 'playerId', 'plusMinus', 'points', 'pointsPerGame', 'positionCode', 'ppGoals', 'ppPoints', 'seasonId', 'shGoals', 'shPoints', 'shootingPct', 'shootsCatches', 'shots', 'skaterFullName', 'teamAbbrevs', 'timeOnIcePerGame']


KeyError: "['playerName'] not in index"

In [4]:
# Advanced Query Examples and Custom Filters

def custom_query(endpoint, filters=None, season="20232024", limit=100):
    """
    Make custom queries with specific filters
    
    Args:
        endpoint (str): API endpoint (e.g., "skater/summary", "team/summary")
        filters (dict): Custom filters to apply
        season (str): Season to query
        limit (int): Number of records to return
    
    Returns:
        pandas.DataFrame: Query results
    """
    # Base filter for regular season games
    base_filter = f"gameTypeId=2 and seasonId<={season}"
    
    # Add custom filters if provided
    if filters:
        custom_filter = " and ".join([f"{k}={v}" for k, v in filters.items()])
        full_filter = f"{base_filter} and {custom_filter}"
    else:
        full_filter = base_filter
    
    params = {
        "cayenneExp": full_filter,
        "limit": limit
    }
    
    data = query_nhl_api(endpoint, params)
    
    if data and 'data' in data:
        df = pd.DataFrame(data['data'])
        print(f"Custom query returned {len(df)} records")
        return df
    else:
        print("Custom query failed")
        return pd.DataFrame()

# Example: Get defensemen with high point totals
print("Advanced Query Examples")
print("=" * 50)

# Query for defensemen with specific criteria
defensemen_stats = custom_query(
    endpoint="skater/summary",
    filters={"playerPositionCode": "D"},  # Defensemen only
    limit=30
)

if not defensemen_stats.empty and 'points' in defensemen_stats.columns:
    top_defensemen = defensemen_stats.nlargest(10, 'points')[['playerName', 'points', 'goals', 'assists', 'playerPositionCode']]
    print("\nTop 10 Defensemen by Points:")
    print(top_defensemen)

print("\n" + "="*60)

# Example: Get goalie stats with specific criteria
print("Fetching goalie statistics...")
goalie_stats = get_goalie_stats(season="20232024", report_type="summary", limit=30)

if not goalie_stats.empty:
    print(f"\nGoalie Stats Shape: {goalie_stats.shape}")
    print(f"Columns: {list(goalie_stats.columns)}")
    
    # Show top goalies by save percentage
    if 'savePctg' in goalie_stats.columns:
        top_goalies = goalie_stats.nlargest(10, 'savePctg')[['playerName', 'savePctg', 'wins', 'losses', 'otLosses']]
        print("\nTop 10 Goalies by Save Percentage:")
        print(top_goalies)

print("\n" + "="*60)
print("Available Report Types:")
print("Skaters: summary, bios, faceoffpercentages, realtime, penalties, powerplay, etc.")
print("Teams: summary, faceoffpercentages, penalties, powerplay, etc.")
print("Goalies: summary, advanced, bios, savesByStrength, etc.")
print("\nYou can now run custom queries using the custom_query() function!")


Advanced Query Examples
Error querying API: 500 Server Error: Server Error for url: https://api.nhle.com/stats/rest/en/skater/summary?cayenneExp=gameTypeId%3D2+and+seasonId%3C%3D20232024+and+playerPositionCode%3DD&limit=30
Custom query failed

Fetching goalie statistics...
Retrieved 30 goalie records

Goalie Stats Shape: (30, 23)
Columns: ['assists', 'gamesPlayed', 'gamesStarted', 'goalieFullName', 'goals', 'goalsAgainst', 'goalsAgainstAverage', 'lastName', 'losses', 'otLosses', 'penaltyMinutes', 'playerId', 'points', 'savePct', 'saves', 'seasonId', 'shootsCatches', 'shotsAgainst', 'shutouts', 'teamAbbrevs', 'ties', 'timeOnIce', 'wins']

Available Report Types:
Skaters: summary, bios, faceoffpercentages, realtime, penalties, powerplay, etc.
Teams: summary, faceoffpercentages, penalties, powerplay, etc.
Goalies: summary, advanced, bios, savesByStrength, etc.

You can now run custom queries using the custom_query() function!
